In [50]:
# -------------------------------------------------------------
# Step 1: Import necessary libraries
# -------------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from textblob import TextBlob

In [51]:
# -------------------------------------------------------------
# Step 2: Load the raw dataset
# -------------------------------------------------------------
print("📥 Loading dataset...")
df = pd.read_csv('../data/raw/twcs.csv')  # Pandas DataFrame

📥 Loading dataset...


In [52]:
# -------------------------------------------------------------
# Step 3: Understand structure — schema, datatypes, nulls
# -------------------------------------------------------------
print("\n📊 Data types and null value check:")
print(df.info())

print("\n❓ Any missing values?")
print(df.isnull().sum())



📊 Data types and null value check:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB
None

❓ Any missing values?
tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64


In [53]:
# 1.Filtrar inbound (mensajes enviado a cuentas de empresa)
df = df[df['inbound'] == True].copy()



In [54]:
# Nos enfocaremos en nuestro analisis usando
#registros que contienen mensajes enviados unicamente a la cuenta de applesupport
df = df[df['text'].str.contains("AppleSupport")]

In [55]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
397,697,115854,True,Tue Oct 31 22:31:23 +0000 2017,@AppleSupport The newest update. I️ made sure ...,699,696.0
399,698,115854,True,Tue Oct 31 22:17:40 +0000 2017,@AppleSupport https://t.co/NV0yucs0lB,696,700.0
400,700,115854,True,Tue Oct 31 22:16:56 +0000 2017,@AppleSupport why are my I️’s changing not sho...,698,NaN
402,702,115855,True,Tue Oct 31 22:11:31 +0000 2017,@AppleSupport Tried resetting my settings .. r...,701,703.0
404,704,115855,True,Tue Oct 31 21:59:17 +0000 2017,@AppleSupport This is what it looks like https...,703,705.0
...,...,...,...,...,...,...,...
2811311,2987500,823737,True,Wed Nov 22 01:21:40 +0000 2017,@AppleSupport I updates slack and everything s...,NaN,2987499.0
2811420,2987605,689907,True,Wed Nov 22 02:11:43 +0000 2017,Hey @AppleSupport - not being able to duplicat...,2987604,NaN
2811422,2987607,823765,True,Wed Nov 22 02:17:14 +0000 2017,Yo @AppleSupport is that weird glitch w/ the c...,2987606,NaN
2811484,2987663,823779,True,Wed Nov 22 03:24:02 +0000 2017,What the fuck @AppleSupport my phone keeps ha...,2987662,NaN


In [56]:
# 2. Convertir la columna 'created_at' a tipo datetime
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')

/tmp/ipykernel_13281/883937500.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')


In [57]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
397,697,115854,True,2017-10-31 22:31:23+00:00,@AppleSupport The newest update. I️ made sure ...,699,696.0
399,698,115854,True,2017-10-31 22:17:40+00:00,@AppleSupport https://t.co/NV0yucs0lB,696,700.0
400,700,115854,True,2017-10-31 22:16:56+00:00,@AppleSupport why are my I️’s changing not sho...,698,NaN
402,702,115855,True,2017-10-31 22:11:31+00:00,@AppleSupport Tried resetting my settings .. r...,701,703.0
404,704,115855,True,2017-10-31 21:59:17+00:00,@AppleSupport This is what it looks like https...,703,705.0
...,...,...,...,...,...,...,...
2811311,2987500,823737,True,2017-11-22 01:21:40+00:00,@AppleSupport I updates slack and everything s...,NaN,2987499.0
2811420,2987605,689907,True,2017-11-22 02:11:43+00:00,Hey @AppleSupport - not being able to duplicat...,2987604,NaN
2811422,2987607,823765,True,2017-11-22 02:17:14+00:00,Yo @AppleSupport is that weird glitch w/ the c...,2987606,NaN
2811484,2987663,823779,True,2017-11-22 03:24:02+00:00,What the fuck @AppleSupport my phone keeps ha...,2987662,NaN


In [58]:
# 3. Eliminar filas con fechas inválidas o nulas
df = df[df['created_at'].notna()]

In [59]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
397,697,115854,True,2017-10-31 22:31:23+00:00,@AppleSupport The newest update. I️ made sure ...,699,696.0
399,698,115854,True,2017-10-31 22:17:40+00:00,@AppleSupport https://t.co/NV0yucs0lB,696,700.0
400,700,115854,True,2017-10-31 22:16:56+00:00,@AppleSupport why are my I️’s changing not sho...,698,NaN
402,702,115855,True,2017-10-31 22:11:31+00:00,@AppleSupport Tried resetting my settings .. r...,701,703.0
404,704,115855,True,2017-10-31 21:59:17+00:00,@AppleSupport This is what it looks like https...,703,705.0
...,...,...,...,...,...,...,...
2811311,2987500,823737,True,2017-11-22 01:21:40+00:00,@AppleSupport I updates slack and everything s...,NaN,2987499.0
2811420,2987605,689907,True,2017-11-22 02:11:43+00:00,Hey @AppleSupport - not being able to duplicat...,2987604,NaN
2811422,2987607,823765,True,2017-11-22 02:17:14+00:00,Yo @AppleSupport is that weird glitch w/ the c...,2987606,NaN
2811484,2987663,823779,True,2017-11-22 03:24:02+00:00,What the fuck @AppleSupport my phone keeps ha...,2987662,NaN


In [60]:
# 4.  Eliminar filas con textos vacíos o nulos
df = df[df['text'].notna()]
df = df[df['text'].str.strip() != '']

In [61]:
# 5. Limpiar y normalizar el texto: pasar a minúsculas y quitar espacios
df['text'] = df['text'].str.lower().str.strip()



In [62]:
# 6. Eliminar tweets duplicados según el ID del tweet
df = df.drop_duplicates(subset=['tweet_id'])

In [63]:
#7.  Asegurarse que los IDs de tweet y autor sean strings (útil para análisis posterior)
df['tweet_id'] = df['tweet_id'].astype(str)
df['author_id'] = df['author_id'].astype(str)



In [64]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
397,697,115854,True,2017-10-31 22:31:23+00:00,@applesupport the newest update. i️ made sure ...,699,696.0
399,698,115854,True,2017-10-31 22:17:40+00:00,@applesupport https://t.co/nv0yucs0lb,696,700.0
400,700,115854,True,2017-10-31 22:16:56+00:00,@applesupport why are my i️’s changing not sho...,698,NaN
402,702,115855,True,2017-10-31 22:11:31+00:00,@applesupport tried resetting my settings .. r...,701,703.0
404,704,115855,True,2017-10-31 21:59:17+00:00,@applesupport this is what it looks like https...,703,705.0
...,...,...,...,...,...,...,...
2811311,2987500,823737,True,2017-11-22 01:21:40+00:00,@applesupport i updates slack and everything s...,NaN,2987499.0
2811420,2987605,689907,True,2017-11-22 02:11:43+00:00,hey @applesupport - not being able to duplicat...,2987604,NaN
2811422,2987607,823765,True,2017-11-22 02:17:14+00:00,yo @applesupport is that weird glitch w/ the c...,2987606,NaN
2811484,2987663,823779,True,2017-11-22 03:24:02+00:00,what the fuck @applesupport my phone keeps ha...,2987662,NaN


In [65]:
# 8. Limpiar columnas de referencia a otros tweets
# Reemplazar valores nulos con cadena vacía y convertir a string
df['response_tweet_id'] = df['response_tweet_id'].fillna('').astype(str)
df['in_response_to_tweet_id'] = df['in_response_to_tweet_id'].fillna('').astype(str)

# 9. Reiniciar el índice del DataFrame después de los filtros
df = df.reset_index(drop=True)

# 10. Revisión final del esquema y algunas filas para validar
print(df.info())
print(df.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97153 entries, 0 to 97152
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   tweet_id                 97153 non-null  object             
 1   author_id                97153 non-null  object             
 2   inbound                  97153 non-null  bool               
 3   created_at               97153 non-null  datetime64[ns, UTC]
 4   text                     97153 non-null  object             
 5   response_tweet_id        97153 non-null  object             
 6   in_response_to_tweet_id  97153 non-null  object             
dtypes: bool(1), datetime64[ns, UTC](1), object(5)
memory usage: 4.5+ MB
None
  tweet_id author_id  inbound                created_at  \
0      697    115854     True 2017-10-31 22:31:23+00:00   
1      698    115854     True 2017-10-31 22:17:40+00:00   
2      700    115854     True 2017-10-31 22

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97153 entries, 0 to 97152
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   tweet_id                 97153 non-null  object             
 1   author_id                97153 non-null  object             
 2   inbound                  97153 non-null  bool               
 3   created_at               97153 non-null  datetime64[ns, UTC]
 4   text                     97153 non-null  object             
 5   response_tweet_id        97153 non-null  object             
 6   in_response_to_tweet_id  97153 non-null  object             
dtypes: bool(1), datetime64[ns, UTC](1), object(5)
memory usage: 4.5+ MB


In [93]:
def get_sentiment(text):
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0:
        return 'positive'
    elif analysis.sentiment.polarity == 0:
        return 'neutral'
    else:
        return 'negative'

df['sentiment'] = df['text'].apply(get_sentiment)

In [95]:
#filtramos unicamente mensajes iniciales enviados para evaluar el sentimiento de los 
#mensajes que inician interacciones
#con el objetivo de enfocar mejor la atención de la operacion
df = df[df['in_response_to_tweet_id'].isna() | (df['in_response_to_tweet_id'].str.strip() == '')]

In [102]:
df = df.reset_index(drop=True)


In [103]:
# -------------------------------------------------------------
print("\n💾 Saving cleaned version to interim file (optional step)...")

df.to_csv("../data/interim/twitter_messages_v1.csv", index=False)
print("✅ Data engineering preprocessing complete!")



💾 Saving cleaned version to interim file (optional step)...
✅ Data engineering preprocessing complete!


In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50838 entries, 0 to 50837
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   tweet_id                 50838 non-null  object             
 1   author_id                50838 non-null  object             
 2   inbound                  50838 non-null  bool               
 3   created_at               50838 non-null  datetime64[ns, UTC]
 4   text                     50838 non-null  object             
 5   response_tweet_id        50838 non-null  object             
 6   in_response_to_tweet_id  50838 non-null  object             
 7   sentiment                50838 non-null  object             
dtypes: bool(1), datetime64[ns, UTC](1), object(6)
memory usage: 2.8+ MB
